# Product Scraper - MercadoLibre

Este notebook busca productos en MercadoLibre basado en un archivo Excel de entrada.

## 1. Configuración Inicial

Primero, clonamos el repositorio e instalamos las dependencias:

In [ ]:
!git clone https://github.com/harolCalzada/scraper-tech-challenge.git
%cd scraper-tech-challenge
!pip install -r requirements.txt

## 2. Importar Dependencias

In [ ]:
import sys
sys.path.append('.')

from src.infrastructure.adapters.mercadolibre_scraper import MercadoLibreScraper
from src.infrastructure.adapters.excel_repository import ExcelRepository
from src.domain.services.product_matcher import ProductMatcher

## 3. Inicializar Componentes

In [ ]:
# Configurar rutas de archivos
input_file = 'data/input.xlsx'
output_file = 'data/output.xlsx'

# Crear directorio data si no existe
!mkdir -p data

# Inicializar componentes
scraper = MercadoLibreScraper()
repository = ExcelRepository()
matcher = ProductMatcher()

## 4. Subir Archivo Excel

Sube el archivo Excel con los productos a buscar:

In [ ]:
from google.colab import files
uploaded = files.upload()

# Guardar el archivo subido en el directorio data
with open('data/input.xlsx', 'wb') as f:
    f.write(uploaded[next(iter(uploaded))])

## 5. Ejecutar el Scraper

Ahora buscaremos los productos en MercadoLibre:

In [ ]:
# Cargar productos del archivo Excel
source_products = repository.load_products(input_file)

# Lista para almacenar todos los productos encontrados
all_scraped_products = []

# Buscar cada producto
for product in source_products:
    # Buscar el producto en MercadoLibre
    scraped_products = await scraper.search_product(product)
    
    # Para cada producto encontrado, calcular su similitud
    for scraped_product in scraped_products:
        matcher.is_match(product, scraped_product)
        all_scraped_products.append(scraped_product)

# Guardar resultados en Excel
repository.save_results(output_file, all_scraped_products)

print(f"Found {len(all_scraped_products)} products")
print(f"Results saved to: {output_file}")

## 6. Descargar Resultados

Descarga el archivo Excel con los resultados:

In [ ]:
files.download('data/output.xlsx')